#### 18_model_deployment

##### 1. Notebook purpose


##### Objective

Deploy the Product Defect image classifier to Databricks Model Serving
using a production-oriented inference interface.

The deployed model accepts a Base64-encoded image and returns:

- predicted class index
- predicted class
- confidence
- defect probability
- OK probability

##### Production design

The registered core model and the deployed serving model have different
responsibilities.

##### Core Model Package

```  text
Input:
    preprocessed image tensor [B, 3, 224, 224]

Output:
    logits [B, 2]

```
##### Serving Model Package

``` text

Input:
    Base64-encoded image

Pipeline:
    Base64
        → decode image
        → RGB conversion
        → ResNet18 preprocessing
        → trained ResNet18
        → logits
        → softmax
        → class mapping

Output:
    structured prediction

```
The serving package reuses the trained model weights. It does not
retrain the classifier.

##### 2. Imports

In [0]:
import base64
from pathlib import Path

import mlflow
import pandas as pd
import torch
import torchvision

from mlflow import MlflowClient
from mlflow.models import infer_signature

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

from src.project_config import (
    REGISTERED_MODEL_NAME,
    SERVING_ENDPOINT_NAME,
)

from src.vision_model import (
    build_resnet18_classifier,
)

from src.inference import (
    ProductDefectServingModel,
    predict_image,
)

##### Why explicitly import torchvision?

`torchvision` is required by the deployed inference package because the
image preprocessing and ResNet18 implementation depend on torchvision.

The notebook also reads `torchvision.__version__` when constructing the
serving environment.

Notebook imports and serving dependencies are related but are not the
same concept:

- Notebook imports = packages directly referenced by this notebook.
- Serving dependencies = packages required by the complete deployed
  inference package.

##### 3. MLflow and registry configuration

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

client = MlflowClient()

workspace_client = WorkspaceClient()

print(
    "Registered model:",
    REGISTERED_MODEL_NAME,
)

print(
    "Serving endpoint:",
    SERVING_ENDPOINT_NAME,
)

##### 4. Identify the source core model

In [0]:
#Version 1 is our governed core model package.

SOURCE_MODEL_VERSION = "1"

source_model_version = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=SOURCE_MODEL_VERSION,
    )
)

print(
    "Source model version:",
    source_model_version.version,
)

print(
    "Source packaging run:",
    source_model_version.run_id,
)

print(
    "Source tags:",
    source_model_version.tags,
)

##### 5. Recover training lineage

In [0]:
RESNET18_RUN_ID = (
    source_model_version.tags.get(
        "source_training_run_id"
    )
)

if not RESNET18_RUN_ID:
    raise ValueError(
        "Version 1 does not contain "
        "'source_training_run_id'."
    )

SOURCE_PACKAGING_RUN_ID = (
    source_model_version.run_id
)

print(
    "Training run:",
    RESNET18_RUN_ID,
)

print(
    "Core packaging run:",
    SOURCE_PACKAGING_RUN_ID,
)

**Why this matters**

The registered model preserves lineage between multiple MLflow objects.

``` text

Training Run
    ↓
learned the model weights

Core Packaging Run
    ↓
packaged those weights with signature and input example

Registered Model Version 1
    ↓
governed model identity

```
The `source_training_run_id` tag allows this independently runnable
notebook to recover the original training run without relying on another
notebook's Python variables.

That is exactly why adding that tag in Notebook 17 was worthwhile.

##### 6. Load the trained model

In [0]:
device = torch.device("cpu")

model = model.to(device)
model.eval()

print(
    "Inference device:",
    device,
)

**Why CPU?**

The initial serving configuration uses CPU because ResNet18 is a
relatively small model and this project does not yet have a measured
latency or throughput requirement that justifies GPU serving.

Serving compute should ultimately be selected using:

- latency requirements
- throughput
- concurrency
- model size
- cost

It should not be selected simply because GPU may have been used during
training.

##### 7. Define class names

Use the canonical training class order:

CLASS_NAMES = [
    "def_front",
    "ok_front",
]

print(
    "Class names:",
    CLASS_NAMES,
)

This corresponds to:

- 0 → def_front
- 1 → ok_front

##### 8. Create the serving wrapper

In [0]:
serving_model = (
    ProductDefectServingModel(
        model=model,
        class_names=CLASS_NAMES,
    )
)

##### 9. Select a real image for interface validation

In [0]:
TEST_IMAGE_PATH = Path(
    "/Volumes/"
    "dbw_agentic_ai_dev/"
    "product_defect_ai/"
    "product_defect_data/"
    "raw/casting_data/"
    "test/def_front/"
    "cast_def_0_1059.jpeg"
)

if not TEST_IMAGE_PATH.exists():
    raise FileNotFoundError(
        f"Test image not found: "
        f"{TEST_IMAGE_PATH}"
    )

print(
    "Test image:",
    TEST_IMAGE_PATH,
)

##### 10. Convert the image to Base64

In [0]:
with open(
    TEST_IMAGE_PATH,
    "rb",
) as image_file:

    image_base64 = (
        base64.b64encode(
            image_file.read()
        )
        .decode("utf-8")
    )

print(
    "Base64 length:",
    len(image_base64),
)

##### 11. Build an example serving request

In [0]:
example_request = pd.DataFrame({
    "image_base64": [
        image_base64
    ]
})

display(
    example_request[
        ["image_base64"]
    ].assign(
        image_base64=lambda df:
            df["image_base64"]
            .str.slice(0, 40)
            + "..."
    )
)

##### 12. Validate the wrapper locally

In [0]:
#Before MLflow packaging, validate the actual serving wrapper.

serving_model.load_context(
    context=None
)

local_serving_prediction = (
    serving_model.predict(
        context=None,
        model_input=example_request,
    )
)

display(
    local_serving_prediction
)

#This test proves the complete local inference pipeline works before deployment.

##### 13. Infer the serving signature

In [0]:
serving_signature = (
    infer_signature(
        example_request,
        local_serving_prediction,
    )
)

print(
    serving_signature
)

##### 14. Normalize serving dependency versions

In [0]:
def serving_package_version(
    version: str,
) -> str:
    """
    Remove local Python package build metadata.

    Example:
        2.7.0+cpu -> 2.7.0
    """
    return version.split("+")[0]


TORCH_VERSION = (
    serving_package_version(
        torch.__version__
    )
)

TORCHVISION_VERSION = (
    serving_package_version(
        torchvision.__version__
    )
)

print(
    "Notebook Torch:",
    torch.__version__,
)

print(
    "Serving Torch:",
    TORCH_VERSION,
)

print(
    "Notebook Torchvision:",
    torchvision.__version__,
)

print(
    "Serving Torchvision:",
    TORCHVISION_VERSION,
)

print(
    "MLflow:",
    mlflow.__version__,
)

print(
    "Pandas:",
    pd.__version__,
)

##### Production lesson

Local build metadata such as `+cpu` should not automatically be copied
into a serving environment requirement.

The notebook environment contained:

    torch==2.7.0+cpu

but the Databricks serving package repository exposed:

    torch==2.7.0

The first serving deployment therefore failed during dependency
resolution.

This was a packaging/environment failure, not a model-quality failure.

##### 15. Define serving dependencies

In [0]:
SERVING_REQUIREMENTS = [
    f"mlflow=={mlflow.__version__}",
    f"torch=={TORCH_VERSION}",
    (
        f"torchvision=="
        f"{TORCHVISION_VERSION}"
    ),
    f"pandas=={pd.__version__}",
    "Pillow",
]

print(
    "Serving requirements:"
)

for requirement in SERVING_REQUIREMENTS:
    print(
        " -",
        requirement,
    )

##### 16. Create the serving packaging run

In [0]:
if mlflow.active_run() is not None:
    mlflow.end_run()


with mlflow.start_run(
    run_name=(
        "resnet18_product_defect_"
        "serving_package"
    )
) as packaging_run:

    SERVING_PACKAGING_RUN_ID = (
        packaging_run.info.run_id
    )

    mlflow.set_tags({
        "model_type": (
            "product_defect_serving_wrapper"
        ),
        "purpose": (
            "databricks_model_serving"
        ),
        "source_training_run_id": (
            RESNET18_RUN_ID
        ),
        "source_packaging_run_id": (
            SOURCE_PACKAGING_RUN_ID
        ),
        "source_registered_model": (
            REGISTERED_MODEL_NAME
        ),
        "source_registered_model_version": (
            SOURCE_MODEL_VERSION
        ),
        "interface": (
            "base64_image"
        ),
    })

    serving_model_info = (
        mlflow.pyfunc.log_model(
            name="model",
            python_model=serving_model,
            signature=serving_signature,
            input_example=example_request,
            code_paths=[
                "src",
            ],
            pip_requirements=(
                SERVING_REQUIREMENTS
            ),
        )
    )

    SERVING_MODEL_URI = (
        serving_model_info.model_uri
    )


print(
    "Serving packaging run:",
    SERVING_PACKAGING_RUN_ID,
)

print(
    "Serving model URI:",
    SERVING_MODEL_URI,
)

**Important MLflow lesson**

Use: serving_model_info.model_uri

rather than manually constructing a model URI.

With newer MLflow behavior, logged models have their own managed model identity, so using the returned URI is safer than assuming the storage structure.

##### 17. Reload the packaged model

In [0]:
packaged_serving_model = (
    mlflow.pyfunc.load_model(
        SERVING_MODEL_URI
    )
)

packaged_prediction = (
    packaged_serving_model.predict(
        example_request
    )
)

display(
    packaged_prediction
)

##### 18. Verify local/package prediction consistency

In [0]:
assert (
    packaged_prediction.loc[
        0,
        "predicted_class",
    ]
    ==
    local_serving_prediction.loc[
        0,
        "predicted_class",
    ]
)

assert (
    packaged_prediction.loc[
        0,
        "predicted_class_index",
    ]
    ==
    local_serving_prediction.loc[
        0,
        "predicted_class_index",
    ]
)

print(
    "Local and packaged predictions "
    "are consistent."
)

##### 19. Register the serving package

In [0]:
registered_serving_model = (
    mlflow.register_model(
        model_uri=SERVING_MODEL_URI,
        name=REGISTERED_MODEL_NAME,
    )
)

SERVING_MODEL_VERSION = str(
    registered_serving_model.version
)

print(
    "Serving model version:",
    SERVING_MODEL_VERSION,
)

##### 20. Add serving-model description

In [0]:
serving_description = """
Serving-oriented Product Defect image classifier.

Architecture:
ResNet18 binary image classifier.

Input:
Base64-encoded product image.

Inference pipeline:
Base64 decode
→ RGB conversion
→ ResNet18 preprocessing
→ trained ResNet18
→ softmax
→ class mapping.

Outputs:
- predicted class index
- predicted class
- confidence
- defect probability
- OK probability

The serving package reuses the trained classifier weights.
No retraining is performed during deployment packaging.

Responsible AI considerations:
- prediction confidence does not guarantee correctness
- false-negative defects were observed during evaluation
- exact development/test duplicates were identified
- production robustness across different cameras, lighting,
  equipment, and manufacturing environments has not been
  established
- human oversight may be required depending on operational risk
""".strip()


client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=SERVING_MODEL_VERSION,
    description=serving_description,
)

##### 21. Add serving-model lineage tags

In [0]:
serving_version_tags = {
    "task": (
        "image_classification"
    ),
    "architecture": (
        "resnet18"
    ),
    "deployment_role": (
        "serving_wrapper"
    ),
    "interface": (
        "base64_image"
    ),
    "preprocessing": (
        "resnet18_pretrained_eval"
    ),
    "source_model_version": (
        SOURCE_MODEL_VERSION
    ),
    "source_training_run_id": (
        RESNET18_RUN_ID
    ),
}

In [0]:
for key, value in (
    serving_version_tags.items()
):

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=SERVING_MODEL_VERSION,
        key=key,
        value=value,
    )

##### 22. Verify registered serving model

In [0]:
serving_version = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=SERVING_MODEL_VERSION,
    )
)

print(
    "Version:",
    serving_version.version,
)

print(
    "Status:",
    serving_version.status,
)

print(
    "Run ID:",
    serving_version.run_id,
)

print(
    "Tags:",
    serving_version.tags,
)

In [0]:
if (
    str(serving_version.status).upper()
    != "READY"
):
    raise RuntimeError(
        "Registered serving model is "
        "not READY."
    )

##### 23. Define endpoint configuration

In [0]:
endpoint_config = (
    EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=(
                    REGISTERED_MODEL_NAME
                ),
                entity_version=(
                    SERVING_MODEL_VERSION
                ),
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    )
)

Initial serving configuration:

- CPU
- Small workload
- scale to zero enabled

This is appropriate for a learning/development deployment.

Production compute should later be selected using measured latency,
throughput, concurrency, availability, and cost requirements.

##### 24. Determine whether the endpoint exists

In [0]:
endpoint_exists = True

try:
    existing_endpoint = (
        workspace_client
        .serving_endpoints
        .get(
            SERVING_ENDPOINT_NAME
        )
    )

except Exception as exc:

    if "RESOURCE_DOES_NOT_EXIST" in str(exc):
        endpoint_exists = False
        existing_endpoint = None

    else:
        raise


print(
    "Endpoint exists:",
    endpoint_exists,
)

##### 25. Create or update the endpoint

In [0]:
if not endpoint_exists:

    print(
        "Creating serving endpoint..."
    )

    workspace_client.serving_endpoints.create_and_wait(
        name=SERVING_ENDPOINT_NAME,
        config=endpoint_config,
    )

else:

    print(
        "Updating existing serving endpoint "
        f"to model version "
        f"{SERVING_MODEL_VERSION}..."
    )

    workspace_client.serving_endpoints.update_config_and_wait(
        name=SERVING_ENDPOINT_NAME,
        served_entities=(
            endpoint_config.served_entities
        ),
    )

##### 26. Verify endpoint state

In [0]:
endpoint = (
    workspace_client
    .serving_endpoints
    .get(
        SERVING_ENDPOINT_NAME
    )
)

print(
    "Endpoint:",
    endpoint.name,
)

print(
    "Ready:",
    endpoint.state.ready,
)

print(
    "Config update:",
    endpoint.state.config_update,
)

In [0]:
for entity in (
    endpoint.config.served_entities
):

    print(
        "Model:",
        entity.entity_name,
    )

    print(
        "Version:",
        entity.entity_version,
    )

    print(
        "Workload size:",
        entity.workload_size,
    )

    print(
        "Scale to zero:",
        entity.scale_to_zero_enabled,
    )

##### 27. 🔍 XAI and ⚖️ RAI deployment considerations

##### 🔍 Explainable AI (XAI)

Deployment does not make the model more explainable.

Earlier Grad-CAM analysis showed where the ResNet18 model was spatially
responsive for a selected class.

Grad-CAM should be interpreted as supporting evidence about model
behavior, not verified defect localization.

The serving endpoint currently returns predictions and probabilities.
It does not return Grad-CAM explanations.


##### ⚖️ Responsible AI (RAI)

A successfully deployed endpoint does not prove that a model is safe
for unrestricted production use.

Important limitations identified during this project include:

1. Three defect images were classified as OK on the original test set.

2. One incorrect prediction had approximately 98% confidence.

3. Confidence therefore must not be interpreted as a guarantee of
   correctness.

4. Exact duplicate analysis identified overlap between development and
   test data.

5. The non-duplicate test subset remained strong, but exact hashing does
   not rule out near duplicates or images from the same physical
   products or manufacturing sequence.

6. Robustness has not been established across different:
   - cameras
   - lighting conditions
   - manufacturing equipment
   - product batches
   - production environments

7. Human review requirements and operating thresholds should be based
   on real operational risks and costs rather than selected arbitrarily.

##### 28. Production incident lesson

#####  Production Lesson — Serving Dependency Resolution

During the first serving deployment attempt, the packaged environment
contained:

    torch==2.7.0+cpu
    torchvision==0.22.0+cpu

The Databricks serving environment could resolve the standard package
versions:

    torch==2.7.0
    torchvision==0.22.0

but not the notebook-specific `+cpu` build variants.

As a result, the serving container failed during dependency resolution.

##### Root cause

The notebook runtime package version was copied directly into the
serving environment specification.

##### Correction

Local build metadata was removed before constructing serving
requirements:

    2.7.0+cpu → 2.7.0

##### Important lesson

The failure occurred in the deployment environment.

It did NOT mean:

- the ResNet18 model was incorrectly trained
- the learned weights were invalid
- the Base64 inference interface was incorrect
- the prediction logic was incorrect

The same trained model and inference pipeline were repackaged with
portable dependencies and deployed successfully.

No retraining was required.

##### 29. Model-versioning lesson


Model versions can change for reasons other than retraining.

In this project:

Version 1
-     Core model package
-     tensor → ResNet18 → logits

Version 2
-     First serving-oriented package
-     Base64 → preprocessing → model → structured prediction
-     Deployment failed because of environment dependency resolution.

Version 3
-     Same learned weights
-     Same inference logic
-     Same Base64 serving contract
-     Corrected serving dependencies
-     Successfully deployed.

Therefore:

-     new model version ≠ necessarily new trained model

A new version can represent a change to:

- inference interface
- preprocessing
- dependencies
- packaging
- metadata
- serving behavior

while preserving the same learned model weights.

##### 30. Final architecture

``` text

                    TRAINING
                       │
                       ▼
             Fine-tuned ResNet18
                       │
                       ▼
               MLflow Training Run
                  3f9b44...
                       │
                       │ lineage
                       ▼
              CORE MODEL PACKAGE
                  Version 1
                       │
                       │ same learned weights
                       ▼
             SERVING MODEL PACKAGE
                  Version 3
                       │
          ┌────────────┴────────────┐
          │                         │
       Model                    Environment
          │                         │
      ResNet18               MLflow 3.0.1
          │                  Torch 2.7.0
      inference.py           Torchvision 0.22.0
          │                  Pandas 2.2.3
      preprocessing          Pillow
          │
          └────────────┬────────────┘
                       │
                       ▼
              Databricks Serving
                       │
                       ▼
       product-defect-classifier-endpoint
                       │
                       ▼
                     READY

```

##### And at inference time:

``` text

Client application
       │
       │ Base64 image
       ▼
Serving endpoint
       │
       ▼
decode_base64_image()
       │
       ▼
PIL RGB image
       │
       ▼
ResNet18 preprocessing
       │
       ▼
Tensor [1,3,224,224]
       │
       ▼
Fine-tuned ResNet18
       │
       ▼
logits [1,2]
       │
       ▼
softmax
       │
       ▼
class mapping
       │
       ▼
{
  predicted_class_index,
  predicted_class,
  confidence,
  defect_probability,
  ok_probability
}

```

##### 31. Key learnings


1. A trained model is only one component of a production inference
   system.

2. The inference pipeline converts real application input into
   model-ready input and converts model output into application-friendly
   predictions.

3. The core model package and serving model package can expose different
   interfaces while using the same learned weights.

4. Base64 is a transport representation for image bytes; it is not a
   machine-learning feature representation.

5. Production preprocessing must remain consistent with the
   preprocessing used during model development.

6. MLflow signatures define and document the expected model interface.

7. MLflow model packaging includes more than weights:
   model + code + dependencies + signature + metadata.

8. Model lineage should be stored explicitly rather than relying on
   notebook execution state.

9. `source_training_run_id` allows the serving package to be traced back
   to the run that learned the weights.

10. Notebook runtime package versions are not automatically valid
    serving-environment requirements.

11. Local build metadata such as `+cpu` may need to be normalized for
    portable deployment environments.

12. A serving-environment failure does not necessarily indicate a model
    failure.

13. A new registered model version does not necessarily mean the model
    was retrained.

14. Existing endpoints should be updated deliberately when deploying a
    newer model version rather than unnecessarily creating new
    endpoints.

15. CPU/GPU serving decisions should be based on measured production
    requirements rather than the compute used during training.

16. A READY endpoint proves successful deployment; it does not prove
    production reliability, safety, fairness, or robustness.

17. 🔍 XAI helps investigate why the model produced a prediction.

18. ⚖️ RAI asks whether that prediction can be appropriately trusted and
    used in the intended operational setting.

##### 32. Notebook conclusion


The Product Defect ResNet18 classifier has now moved from a trained
machine-learning model to a deployable inference service.

The final deployment separates four important responsibilities:

    Training
        → learn model weights

    MLflow Packaging
        → package model, code, dependencies, signature, and metadata

    Unity Catalog Model Registry
        → provide governed model identity, versioning, and lineage

    Databricks Model Serving
        → expose the inference pipeline through a live endpoint

The production serving interface accepts a Base64-encoded image,
performs the required ResNet18 preprocessing, executes the trained
classifier, and returns structured prediction results.

The deployment process also demonstrated an important real-world MLOps
lesson: deployment failures can originate from packaging and dependency
management rather than model training.

The initial serving package failed because notebook-specific PyTorch
build metadata was not portable to the serving environment. The same
trained weights and inference logic were subsequently packaged with
portable dependency versions and deployed successfully without
retraining.

The active serving endpoint is:

    product-defect-classifier-endpoint

and the successfully deployed serving model is Version 3.

The next notebook will validate the deployed system through endpoint
requests, including successful predictions, known false-negative cases,
response-schema validation, and invalid-input behavior.